# AutoDocIQ Code Explanation

Welcome to the **AutoDocIQ** code walkthrough! This Jupyter Notebook serves as an interactive and comprehensive guide to the underlying codebase of the AutoDocIQ Autonomous Document Intelligence System. Here, we break down each primary file, explaining the logic and providing the full implementation for analysis.

--- 
## 1. System Entrypoint (`run.py`)

The `run.py` script serves as the unified orchestrator for the system's runtime check and launch. It performs two key operations:
- **Dependency Validation**: On boot, it checks if the necessary packages (FastAPI, Uvicorn, LangChain, ChromaDB, Pydantic, Pandas, and PyPDF) are already installed. If any imports fail, it automatically runs `pip install -r requirements.txt` to install the requirements.
- **Backend Server Initialization**: Once dependencies are resolved, it uses Uvicorn to boot up the FastAPI app instance on port `8000` with hot-reloading enabled.

In [ ]:
# Code implementation for run.py
import os
import sys
import subprocess

def check_and_install_dependencies():
    print("Checking dependencies...")
    try:
        import fastapi
        import uvicorn
        import langchain
        import chromadb
        import pydantic
        import pandas
        import pypdf
        print("All dependencies are already installed.")
    except ImportError as e:
        print(f"Missing dependency detected: {e.name}. Installing from requirements.txt...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
            print("Dependencies installed successfully.")
        except Exception as err:
            print(f"Failed to install dependencies: {err}")
            sys.exit(1)

def run_server():
    print("Starting FastAPI Backend Server on port 8000...")
    try:
        import uvicorn
        uvicorn.run("backend.main:app", host="0.0.0.0", port=8000, reload=True)
    except KeyboardInterrupt:
        print("\nServer stopped by user.")
    except Exception as e:
        print(f"Error starting server: {e}")

if __name__ == "__main__":
    sys.path.insert(0, os.path.abspath(os.path.dirname(__file__)))
    # Note: Dependencies checks are commented out below to prevent execution locks when importing in notebooks
    # check_and_install_dependencies()
    # run_server()

--- 
## 2. Configuration Layer (`backend/config.py`)

The `config.py` configuration loader establishes paths, loads persistency configurations, and toggles execution modes:
- **Directory Setup**: Dynamically resolves system directories like `BASE_DIR`, `uploads/`, `chroma/`, and the local `settings.json` path.
- **Modes Validation**: Tries loading settings from `data/settings.json`. If an OpenAI API Key is present, it selects **Live AI Mode**; otherwise, it boots into **Graceful Mock Mode**.
- **Dynamic Persistence**: Exposes `save_settings` to update configuration parameters dynamically when modified via the frontend settings modal.

In [ ]:
# Code implementation for backend/config.py
import os
import json

class Settings:
    def __init__(self):
        self.BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
        self.UPLOAD_DIR = os.path.join(self.BASE_DIR, "data", "uploads")
        self.CHROMA_DIR = os.path.join(self.BASE_DIR, "data", "chroma")
        self.SETTINGS_FILE = os.path.join(self.BASE_DIR, "data", "settings.json")
        
        os.makedirs(self.UPLOAD_DIR, exist_ok=True)
        os.makedirs(self.CHROMA_DIR, exist_ok=True)
        
        self.OPENAI_API_KEY = ""
        self.MOCK_MODE = False
        self.load_settings()

    def load_settings(self):
        if os.path.exists(self.SETTINGS_FILE):
            try:
                with open(self.SETTINGS_FILE, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    self.OPENAI_API_KEY = data.get("OPENAI_API_KEY", "")
                    self.MOCK_MODE = data.get("MOCK_MODE", False)
                    print(f"Loaded settings: MOCK_MODE={self.MOCK_MODE}")
                    return
            except Exception as e:
                print(f"Error loading settings: {e}")

        self.OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
        if not self.OPENAI_API_KEY:
            self.MOCK_MODE = True
        else:
            self.MOCK_MODE = False

    def save_settings(self, api_key: str, mock_mode: bool):
        self.OPENAI_API_KEY = api_key
        self.MOCK_MODE = mock_mode
        try:
            with open(self.SETTINGS_FILE, "w", encoding="utf-8") as f:
                json.dump({
                    "OPENAI_API_KEY": self.OPENAI_API_KEY,
                    "MOCK_MODE": self.MOCK_MODE
                }, f, indent=2)
            print("Successfully saved settings.json")
        except Exception as e:
            print(f"Failed to save settings: {e}")

--- 
## 3. Data Schema Models (`backend/schemas.py`)

The serialization layer uses Pydantic to enforce type safety, validation boundaries, and structured JSON parsing specifications:
- `ContractEntitySchema`: Structured model representing extracted contract terms (parties, dates, value, governing law, jurisdiction limits).
- `InvoiceEntitySchema` & `InvoiceLineItem`: Structured models validating invoices (invoice number, supplier, buyer, line items total, tax, currency).
- `ClauseClassification`: Model validating segmented clause compliance (text segment, risk level, confidence rating, legal justification).
- `QueryRequest` & `QueryResponse`: Handle interface communication constraints for chat and semantic context citations.

In [ ]:
# Code implementation for backend/schemas.py
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

class ContractEntitySchema(BaseModel):
    contract_type: str = Field(description="Type of contract, e.g., NDA, Service Agreement.")
    parties: List[str] = Field(default=[], description="Primary legal entities involved.")
    effective_date: Optional[str] = Field(None)
    expiration_date: Optional[str] = Field(None)
    contract_value: Optional[str] = Field(None)
    governing_law: Optional[str] = Field(None)
    indemnity_limit: Optional[str] = Field(None)
    termination_notice_period: Optional[str] = Field(None)
    confidentiality_duration: Optional[str] = Field(None)

class InvoiceLineItem(BaseModel):
    description: str
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    amount: Optional[float] = None

class InvoiceEntitySchema(BaseModel):
    invoice_number: Optional[str] = None
    supplier: Optional[str] = None
    buyer: Optional[str] = None
    issue_date: Optional[str] = None
    due_date: Optional[str] = None
    line_items: List[InvoiceLineItem] = Field(default=[])
    tax_amount: Optional[float] = None
    total_amount: Optional[float] = None
    currency: Optional[str] = "USD"

class ClauseClassification(BaseModel):
    clause_type: str
    text_segment: str
    confidence: float
    risk_level: str
    reasoning: str

--- 
## 4. Vector Storage System & TF-IDF Fallback (`backend/vectorstore.py`)

The storage and vector search framework is designed for reliability across environments:
- **Chunk Separation**: Splits parsed documents into 1000-character portions with a 150-character overlay using `RecursiveCharacterTextSplitter`.
- **Dual-Store Engines**:
  - `ChromaVectorStore`: Standard database client storing and querying dense embeddings cosine similarities locally.
  - `InMemoryVectorStore`: Fallback pure-Python engine implementing a classic TF-IDF token frequency Jaccard correlation matcher, bypassing system binary compilation boundaries if ChromaDB is missing.

In [ ]:
# Code implementation for backend/vectorstore.py
import re
from typing import List, Dict, Any

class InMemoryVectorStore:
    """Fallback term-frequency TF-IDF similarity matcher."""
    def __init__(self):
        self.documents = {}

    def add_chunks(self, doc_id: str, chunks: List[str], filename: str):
        if doc_id not in self.documents:
            self.documents[doc_id] = []
        for i, chunk in enumerate(chunks):
            self.documents[doc_id].append({
                "id": f"{doc_id}_chunk_{i}",
                "text": chunk,
                "metadata": {"filename": filename, "chunk_index": i}
            })

    def query(self, doc_id: str, query_text: str, limit: int = 3) -> List[Dict[str, Any]]:
        if doc_id not in self.documents or not self.documents[doc_id]:
            return []
        
        chunks = self.documents[doc_id]
        def get_terms(text: str) -> set:
            return set(re.findall(r'\w+', text.lower()))

        q_terms = get_terms(query_text)
        if not q_terms:
            return chunks[:limit]

        scored_chunks = []
        for chunk in chunks:
            c_terms = get_terms(chunk["text"])
            intersection = q_terms.intersection(c_terms)
            union = q_terms.union(c_terms)
            score = len(intersection) / len(union) if union else 0.0
            scored_chunks.append((chunk, score))
        
        scored_chunks.sort(key=lambda x: x[1], reverse=True)
        return [{
            "text": chunk["text"],
            "score": round(1.0 - score, 4),
            "metadata": chunk["metadata"]
        } for chunk, score in scored_chunks[:limit]]

--- 
## 5. LLM Multi-Agent Orchestration & Heuristics Fallbacks (`backend/agents.py`)

This coordinates the different expert agents. It supports both **Online** (GPT-4o via LangChain) and **Offline** (heuristic fallback) execution:
- `RouterAgent`: Classifies client queries to route them correctly (`extraction` vs `classification` vs `qa`).
- `ExtractionAgent`: Parses structured metrics from document context into Pydantic models.
- `ClassificationAgent`: Evaluates text snippets to detect risk boundaries in contract parameters.
- `SemanticQ&AAgent`: Runs semantic search over context chunks and references source chunks in answers.

In [ ]:
# Code implementation for backend/agents.py (Mock Heuristic logic snippet)
import re
from typing import List, Dict, Any, Tuple

class MockAgents:
    @staticmethod
    def route_request(query: str) -> Tuple[str, List[str]]:
        logs = ["RouterAgent: Analyzing query intent..."]
        q_lower = query.lower()
        if any(w in q_lower for w in ["extract", "parties", "value", "dates", "schema", "entity"]):
            logs.append("RouterAgent: Routing query task to [ExtractionAgent].")
            return "extraction", logs
        elif any(w in q_lower for w in ["clause", "indemnity", "termination", "governing law", "liability"]):
            logs.append("RouterAgent: Routing query task to [ClassificationAgent].")
            return "classification", logs
        else:
            logs.append("RouterAgent: Routing query task to [SemanticQ&AAgent].")
            return "qa", logs

    @staticmethod
    def extract_entities(content: str, file_type: str) -> Tuple[Dict[str, Any], List[str]]:
        logs = ["ExtractionAgent: Commencing rule-based entity extraction..."]
        content_lower = content.lower()
        
        if "invoice" in content_lower or "bill to" in content_lower:
            logs.append("ExtractionAgent: Matching invoice template.")
            result = {
                "schema_type": "invoice",
                "data": {
                    "invoice_number": "INV-2026-904",
                    "supplier": "Alpha Cloud Solutions LLC",
                    "buyer": "Apex Global Enterprises Inc",
                    "total_amount": 17118.00,
                    "currency": "USD"
                }
            }
        else:
            logs.append("ExtractionAgent: Matching contract template.")
            result = {
                "schema_type": "contract",
                "data": {
                    "contract_type": "Mutual Non-Disclosure Agreement",
                    "parties": ["TechNexus Solutions Inc.", "AeroSpace Global Corp."],
                    "effective_date": "2026-05-10",
                    "governing_law": "California, USA"
                }
            }
        return result, logs

--- 
## 6. API Routing & Controller Setup (`backend/main.py`)

The FastAPI application mounts static assets and configures CORS. It exposes these core endpoints:
- `GET /api/health`: Monitors database, environment modes, and OpenAI configuration parameters.
- `POST /api/settings`: Updates API key settings and resets the live execution orchestrator.
- `GET /api/documents`: Lists all uploaded files and metadata summaries.
- `POST /api/upload`: Handles chunking pipelines and triggers the agent summaries on document uploads.
- `POST /api/documents/{id}/query`: Executes RAG queries and compiles citation chunks.
- `GET /api/documents/{id}/export`: Exports compliance summaries to CSV or JSON formats.
- `DELETE /api/documents/{id}`: Cleans files from disk and purges collection vectors.

In [ ]:
# Code snippet illustrating upload and seeding controller logic from backend/main.py
# Note: Full server controllers run on fastapi app wrapper
print("FastAPI app loaded on boot:")
print(" - Mounted: '/' static assets directory mapping to index.html")
print(" - Seeds: Loaded sample_nda.txt & sample_invoice.txt in uploads on startup")